In [ ]:
#Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

In [ ]:
#Load and Scale Data ---
df = pd.read_csv('../data/tmQM_y.csv')
features = ['HL_Gap', 'Dipole_M', 'Molecular Size', 'MND', 'q']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
#Run DBSCAN ---
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(X_scaled)
df['DBSCAN_Label'] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)
print(f"Estimated number of clusters: {n_clusters}")
print(f"Estimated number of noise points: {n_noise}")

In [ ]:
#Validation Metrics ---
mask = labels != -1
X_clustered = X_scaled[mask]
labels_clustered = labels[mask]

# Sampling to avoid memory constraints during metric calculation
sample = X_clustered[:10000]
sample_labels = labels_clustered[:10000]

if len(set(sample_labels)) > 1:
    print(f"Silhouette Score: {silhouette_score(sample, sample_labels):.3f}")
    print(f"Davies-Bouldin Score: {davies_bouldin_score(sample, sample_labels):.3f}")
    print(f"Calinski-Harabasz Score: {calinski_harabasz_score(sample, sample_labels):.3f}")
else:
    print("Not enough clusters detected to compute validation metrics.")

In [ ]:
#Export Data ---
df.to_csv('../data/DBSCAN_clustering_results.csv', index=False)
print("Clustering results successfully saved to DBSCAN_clustering_results.csv")

In [ ]:
#3D Visualization ---
# Fixed the variable mapping to use actual column names instead of '1', '2', '3'
fig = px.scatter_3d(
    df,
    x='HL_Gap',
    y='Dipole_M',
    z='Molecular Size',
    color='DBSCAN_Label',
    title="DBSCAN Clustering 3D Projection",
    opacity=0.7
)
fig.show()